# Chapter 06 — Byte Pair Encoding Implementation

Tokenization is the bridge between human text and neural networks. Models don't see words or characters — they see numbers. The tokenizer converts raw text into a sequence of integer IDs, and this seemingly simple preprocessing step has profound effects on model performance, vocabulary size, and even what languages the model can handle well.

### Glossary / Glossário

• BPE — Byte Pair Encoding, tokenization algorithm that iteratively merges frequent character pairs. / Byte Pair Encoding, algoritmo de tokenização que iterativamente mescla pares de caracteres frequentes.
• tokenization — process of splitting raw text into tokens (integer IDs) for model input. / Processo de dividir texto bruto em tokens (IDs inteiros) para entrada do modelo.
• subword — token that is part of a word, e.g. "un" + "happi" + "ness". / Token que é parte de uma palavra, ex: "in" + "felicidade".
• vocabulary size — total number of unique tokens the model can recognize. / Número total de tokens únicos que o modelo reconhece.
• UTF-8 — variable-length Unicode encoding standard where some characters use multiple bytes. / Padrão de codificação Unicode de comprimento variável onde alguns caracteres usam múltiplos bytes.

In [ ]:
# === Setup: text to tokenize ===
text = "Tokenização é a ponte entre texto humano e redes neurais. Os modelos não veem palavras."
sequences = [list(text.encode("utf-8"))]  # sequences: list of token sequences (starts as raw bytes, gets shorter with each merge)
initial_len = sum(len(s) for s in sequences)

def get_pair_counts(token_sequences):
    """Count frequency of adjacent token pairs."""
    counts = {}  # counts: frequency of each adjacent token pair in all sequences
    for seq in token_sequences:
        for i in range(len(seq) - 1):
            pair = (seq[i], seq[i + 1])
            counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge_pair(sequences, pair, new_token):
    """Replace all occurrences of pair with new_token."""
    result = []
    for seq in sequences:
        new_seq = []
        i = 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i+1]) == pair:
                new_seq.append(new_token)  # new_token = 256 + i: ID for the newly created merged token
                i += 2
            else:
                new_seq.append(seq[i])
                i += 1
        result.append(new_seq)
    return result

# === Aha: watch compression improve with each merge ===
vocab_size = 276  # vocab_size: target vocab = 256 base bytes + 20 learned merges
print(f"\n{'Merge':>5s} | {'Pair':>15s} | {'Total Tokens':>13s} | {'Compression':>12s}")
print("-" * 52)
for i in range(20):
    counts = get_pair_counts(sequences)  # counts: frequency of each adjacent token pair
    if not counts:
        break
    best_pair = max(counts, key=counts.get)  # best_pair: the most frequent pair — will be merged next
    sequences = merge_pair(sequences, best_pair, 256 + i)
    current_len = sum(len(s) for s in sequences)
    ratio = initial_len / current_len
    print(f"{i:>5d} | {str(best_pair):>15s} | {current_len:>13d} | {ratio:>11.2f}x")
print(f"\nText went from {initial_len} tokens to {current_len} tokens — {ratio:.1f}x compression!")

---

**Course:** Aprenda LLM | **Chapter 06**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.